In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import PowerTransformer
import lightgbm as lgb
from scipy.signal import savgol_filter
import copy

# --- CONFIGURATION & DEVICE ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# --- DATASET PATHS ---
# Asigură-te că datele sunt structurate local în folderul './data'
ROOT_DIR = './data'

labels_path = os.path.join(ROOT_DIR, 'train.csv') # Modifică în 'labels.csv' dacă așa se numește fișierul tău
sample_sub_path = os.path.join(ROOT_DIR, 'test.csv') # Modifică în 'sample_submission.csv' dacă e cazul

if not os.path.exists(labels_path):
    raise RuntimeError(f"Nu găsesc fișierul de antrenare la calea: {labels_path}. Verifică structura folderului 'data/'.")

if os.path.exists(sample_sub_path):
    test_df = pd.read_csv(sample_sub_path)
else:
    test_files = sorted(glob.glob(os.path.join(ROOT_DIR, 'test', '*.npy')))
    test_df = pd.DataFrame({'filename': [os.path.basename(f) for f in test_files]})

# --- LIGHTGBM PART ---
def extract_features(df, root_dir, is_test=False):
    features = []
    labels = []
    for idx, row in df.iterrows():
        fname = row['filename']
        paths = [
            os.path.join(root_dir, 'test' if is_test else 'train', fname),
            os.path.join(root_dir, 'test' if is_test else '', fname),
            os.path.join(root_dir, fname)
        ]
        path = next((p for p in paths if os.path.exists(p)), None)
        if path:
            img = np.load(path).astype('float32')
            center_pixel = img[9, 9, :]
            features.append(center_pixel)
            if not is_test:
                labels.append(int(row['label']) - 1)
        else:
            features.append(np.zeros(48))
            if not is_test: labels.append(0)
    return np.array(features), np.array(labels) if not is_test else None

full_train_df = pd.read_csv(labels_path)
X_all, y_all = extract_features(full_train_df, ROOT_DIR, is_test=False)
X_test, _ = extract_features(test_df, ROOT_DIR, is_test=True)

pt = PowerTransformer(method='yeo-johnson', standardize=True)
X_all_pt = pt.fit_transform(X_all)
X_test_pt = pt.transform(X_test)

lgb_preds = np.zeros((len(X_test), 7))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all_pt, y_all)):
    X_tr, X_val = X_all_pt[train_idx], X_all_pt[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    clf = lgb.train(
        {'objective': 'multiclass', 'num_class': 7, 'metric': 'multi_logloss', 'learning_rate': 0.05, 'num_leaves': 31, 'verbose': -1},
        dtrain, num_boost_round=1000, valid_sets=[dtrain, dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    lgb_preds += clf.predict(X_test_pt) / 5

# --- RESNET PART ---
class SGResNetDataset(Dataset):
    def __init__(self, df, root_dir, augment=False, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.augment = augment
        self.is_test = is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]
            fname = row['filename']
            paths = [
                os.path.join(self.root_dir, 'test' if self.is_test else 'train', fname),
                os.path.join(self.root_dir, fname)
            ]
            path = next((p for p in paths if os.path.exists(p)), paths[0])
            img = np.load(path).astype('float32')
            filtered = savgol_filter(img, window_length=5, polyorder=2, axis=2)
            raw = filtered[9, 9, :]
            raw = (raw - raw.mean()) / (raw.std() + 1e-8)
            pixel = raw.reshape(1, -1)
            if self.augment:
                if np.random.rand() > 0.5: pixel = pixel + np.random.normal(0, 0.02, pixel.shape)
                if np.random.rand() > 0.5: pixel = pixel * np.random.uniform(0.98, 1.02)
            label = int(row['label']) - 1 if not self.is_test else 0
            return torch.tensor(pixel, dtype=torch.float32), label
        except: return torch.zeros((1, 48), dtype=torch.float32), 0

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_c)
        self.skip = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(nn.Conv1d(in_c, out_c, 1, stride, bias=False), nn.BatchNorm1d(out_c))
    def forward(self, x):
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + self.skip(x))

class ResNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_c = 64
        self.conv1 = nn.Conv1d(1, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool1d(3, 2, 1)
        self.l1 = self._layer(64, 2, 1)
        self.l2 = self._layer(128, 2, 2)
        self.l3 = self._layer(256, 2, 2)
        self.fc = nn.Linear(256, 7)
    def _layer(self, out_c, blocks, stride):
        layers = [ResidualBlock(self.in_c, out_c, stride)]
        self.in_c = out_c
        for _ in range(1, blocks): layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.l3(self.l2(self.l1(x)))
        return self.fc(torch.mean(x, dim=2))

resnet_preds = []
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_all), y=y_all)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)

for fold, (train_idx, val_idx) in enumerate(skf.split(full_train_df, full_train_df['label'])):
    print(f"Fold {fold+1}...")
    train_ds = SGResNetDataset(full_train_df.iloc[train_idx], ROOT_DIR, augment=True)
    val_ds = SGResNetDataset(full_train_df.iloc[val_idx], ROOT_DIR)
    t_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
    v_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

    model = ResNet1D().to(device)
    optim_fn = optim.AdamW(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(optim_fn, T_0=10)

    best_acc = 0; best_wts = copy.deepcopy(model.state_dict())

    for ep in range(20): # MAX PERFORMANCE EPOCHS
        model.train()
        for x, y in t_loader:
            optim_fn.zero_grad()
            crit(model(x.to(device)), y.to(device)).backward()
            optim_fn.step()
        sched.step()
        model.eval()
        c=0; t=0
        with torch.no_grad():
            for x, y in v_loader:
                out = model(x.to(device))
                c += (out.argmax(1) == y.to(device)).sum().item()
                t += y.size(0)
        if (c/t) > best_acc: best_acc = c/t; best_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_wts)
    model.eval()
    test_ds = SGResNetDataset(test_df, ROOT_DIR, augment=True, is_test=True)
    t_loader = DataLoader(test_ds, batch_size=128, shuffle=False)
    fold_p = []
    for _ in range(4): # TTA
        ep_p = []
        with torch.no_grad():
            for x, _ in t_loader: ep_p.append(torch.softmax(model(x.to(device)), 1).cpu().numpy())
        fold_p.append(np.concatenate(ep_p))
    resnet_preds.append(np.mean(fold_p, axis=0))

avg_resnet_preds = np.mean(resnet_preds, axis=0)
final_probs = 0.6 * avg_resnet_preds + 0.4 * lgb_preds

QUOTAS = {2: 2850, 7: 2100, 1: 950, 3: 763}
fill_order = [3, 1, 7, 2]

final_labels = np.argmax(final_probs, axis=1) + 1
assigned_mask = np.zeros(len(test_df), dtype=bool)

rare_mask = (final_probs[:, 3] > 0.9) | (final_probs[:, 4] > 0.9)
final_labels[rare_mask] = np.argmax(final_probs[rare_mask], axis=1) + 1
assigned_mask[rare_mask] = True

for cls in fill_order:
    scores = final_probs[:, cls-1]
    scores[assigned_mask] = -1.0
    idx = np.argsort(scores)[::-1][:QUOTAS[cls]]
    final_labels[idx] = cls
    assigned_mask[idx] = True

final_labels[~assigned_mask] = 6

# Salvăm output-ul local în folderul proiectului
output_path = './submission_HYBRID_FINAL.csv'
pd.DataFrame({'filename': test_df['filename'], 'label': final_labels}).to_csv(output_path, index=False)
print(f"DONE! File saved to {output_path}")